# Reading and Writing Data

**Dataset:** `samples.bakehouse.sales_transactions`, `samples.nyctaxi.trips`

**Difficulty:** Hard

**Topics:** `spark.read.format`, `spark.read.csv`, `spark.read.json`, `spark.read.parquet`, `inferSchema`, explicit schema, `header`, `sep`, `partitionBy`, `saveAsTable`, `write.mode`, `mergeSchema`, reading back written files

> All source data comes from the `samples` catalog - no external files needed.
> Intermediate files are written to `/tmp/` which is available in all Databricks environments.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_trips        = spark.table("samples.nyctaxi.trips")

# Base path for all intermediate files in this notebook
BASE = "/tmp/spark_io_practice"


## Problem 1 - Write CSV and Read it Back with inferSchema

Write a sample of `df_trips` (1000 rows) to CSV format at `BASE + "/trips_csv/"` with `header=True`.
Then read it back using `spark.read.format("csv")` with options `header=True` and `inferSchema=True`.

**Expected output columns:** same columns as the original sample

> `inferSchema=True` makes Spark scan the file to detect column types. For large files use an explicit schema instead.

Assign result to: `result_1`

In [ ]:
# Problem 1 - write your solution here
# result_1 = None  # replace this

In [ ]:
# Tests for Problem 1
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
assert len(result_1.columns) > 3, "Expected several columns"
print(f"Problem 1 passed ({cnt} rows, {len(result_1.columns)} columns)")

## Problem 2 - Read CSV with Explicit Schema (no inferSchema)

Write a 500-row sample of `df_transactions` to CSV at `BASE + "/transactions_csv/"`.
Then read it back with an **explicit StructType schema** instead of `inferSchema`.
Define at least 4 columns: `transactionID` (IntegerType), `franchiseID` (IntegerType), `product` (StringType), `totalPrice` (DoubleType).

**Expected output columns:** `transactionID`, `franchiseID`, `product`, `totalPrice` (at minimum)

> Explicit schemas are faster (no scan needed) and more reliable for production pipelines.

Assign result to: `result_2`

In [ ]:
# Problem 2 - write your solution here
# result_2 = None  # replace this

In [ ]:
# Tests for Problem 2
assert result_2 is not None, "result_2 is None"
cols = [c.lower() for c in result_2.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'totalprice' in cols, "Missing column: totalPrice"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
from pyspark.sql import types as T
price_field = [f for f in result_2.schema.fields if f.name.lower() == 'totalprice'][0]
assert isinstance(price_field.dataType, T.DoubleType), "totalPrice should be DoubleType from explicit schema"
print(f"Problem 2 passed ({cnt} rows, explicit schema applied)")

## Problem 3 - Write and Read JSON

Write a 1000-row sample of `df_transactions` to JSON format at `BASE + "/transactions_json/"` with `mode="overwrite"`.
Then read it back with `spark.read.format("json")` and `inferSchema=True`.

**Expected output columns:** same as source

> JSON files store schema inline per record. `inferSchema` on JSON is reliable but slow on very large files.

Assign result to: `result_3`

In [ ]:
# Problem 3 - write your solution here
# result_3 = None  # replace this

In [ ]:
# Tests for Problem 3
assert result_3 is not None, "result_3 is None"
cols = [c.lower() for c in result_3.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 3 passed ({cnt} rows from JSON)")

## Problem 4 - Write and Read Parquet

Write a 2000-row sample of `df_transactions` to Parquet format at `BASE + "/transactions_parquet/"`.
Then read it back with `spark.read.parquet(path)`.

Parquet is a columnar binary format - it preserves schema and column types exactly, and is the most efficient format for Spark.

**Expected output columns:** same as source, types preserved exactly

Assign result to: `result_4`

In [ ]:
# Problem 4 - write your solution here
# result_4 = None  # replace this

In [ ]:
# Tests for Problem 4
assert result_4 is not None, "result_4 is None"
cols = [c.lower() for c in result_4.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
# Parquet preserves types - totalPrice should remain DoubleType
from pyspark.sql import types as T
price_field = [f for f in result_4.schema.fields if f.name.lower() == 'totalprice'][0]
assert isinstance(price_field.dataType, (T.DoubleType, T.FloatType)), "totalPrice should be numeric type"
print(f"Problem 4 passed ({cnt} rows, types preserved by Parquet)")

## Problem 5 - Partitioned Write

Write all of `df_transactions` to Parquet at `BASE + "/transactions_partitioned/"` using `.partitionBy("franchiseID")`.
Then read it back.

Partitioned writes create subdirectories like `franchiseID=1/`, `franchiseID=2/`, etc.
Spark uses partition pruning to skip irrelevant directories when filtering by the partition column.

**Expected output columns:** same as source (franchiseID re-added from partition)

Assign result to: `result_5`

In [ ]:
# Problem 5 - write your solution here
# result_5 = None  # replace this

In [ ]:
# Tests for Problem 5
assert result_5 is not None, "result_5 is None"
cols = [c.lower() for c in result_5.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
franchise_count = result_5.select('franchiseID').distinct().count()
assert franchise_count > 1, "Expected multiple franchise partitions"
print(f"Problem 5 passed ({cnt} rows, {franchise_count} franchise partitions)")

## Problem 6 - Custom Separator CSV (pipe-delimited)

Create a small in-memory DataFrame and write it as a pipe-delimited (`|`) CSV with headers:

```python
df_pipe = spark.createDataFrame(
    [(1, "Alice", 100.0), (2, "Bob", 200.0), (3, "Carol", 300.0)],
    ["id", "name", "value"]
)
```

Write to `BASE + "/pipe_csv/"` with `sep="|"` and `header=True`.
Read it back specifying the same `sep="|"` option.

**Expected output columns:** `id`, `name`, `value`

Assign result to: `result_6`

In [ ]:
# Problem 6 - write your solution here
# result_6 = None  # replace this

In [ ]:
# Tests for Problem 6
assert result_6 is not None, "result_6 is None"
cols = [c.lower() for c in result_6.columns]
assert 'id' in cols, "Missing column: id"
assert 'name' in cols, "Missing column: name"
assert 'value' in cols, "Missing column: value"
cnt = result_6.count()
assert cnt == 3, f"Expected 3 rows, got {cnt}"
print(f"Problem 6 passed ({cnt} rows, pipe-delimited round-trip)")

## Problem 7 - saveAsTable (Managed Table)

Write the first 500 rows of `df_transactions` to a managed Delta table named `transactions_practice_table` using:

```python
.write.format("delta").mode("overwrite").saveAsTable("transactions_practice_table")
```

Then read it back with `spark.table("transactions_practice_table")`.

Managed tables are tracked by the metastore - you can query them by name from any notebook.

**Expected output columns:** same as source

Assign result to: `result_7`

In [ ]:
# Problem 7 - write your solution here
# result_7 = None  # replace this

In [ ]:
# Tests for Problem 7
assert result_7 is not None, "result_7 is None"
cols = [c.lower() for c in result_7.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 7 passed ({cnt} rows from managed table)")

## Problem 8 - Schema Evolution with mergeSchema

Write two DataFrames with **different schemas** to the same Parquet path to demonstrate schema evolution.

Step 1: Write `df_transactions.select("transactionID","franchiseID","totalPrice")` to `BASE + "/schema_evolution/"`.

Step 2: Write `df_transactions.select("transactionID","franchiseID","totalPrice","product")` to the same path with `mode="append"` and `.option("mergeSchema","true")`.

Step 3: Read back with `.option("mergeSchema","true")` - the result should have all 4 columns, with nulls for `product` in older rows.

**Expected output columns:** `transactionID`, `franchiseID`, `totalPrice`, `product`

Assign result to: `result_8`

In [ ]:
# Problem 8 - write your solution here
# result_8 = None  # replace this

In [ ]:
# Tests for Problem 8
assert result_8 is not None, "result_8 is None"
cols = [c.lower() for c in result_8.columns]
assert 'transactionid' in cols, "Missing column: transactionID"
assert 'product' in cols, "Missing: product (schema evolution should add this)"
cnt = result_8.count()
assert cnt > 0, "Got 0 rows"
null_product_count = result_8.filter(F.col('product').isNull()).count()
assert null_product_count > 0, "Expected some rows with null product (from first write without that column)"
print(f"Problem 8 passed ({cnt} rows, {null_product_count} rows with null product from schema evolution)")